# DIEM Hub community management

**Internal use only. Never publish this notebook or copy it to the deployment repository.**

Accounts created on the DIEM Hub belong to the ArcGIS Hub community organization (`hqfao-hub`). The content promised to them is shared with groups in the FAO ArcGIS Online organization (`hqfao`), which accounts cannot join automatically. This notebook bridges the two organizations.

Each run:

1. Reads the members of the Hub community group.
2. Keeps the role table in step: new accounts are added with role 1, rows for accounts that left are deleted, and existing roles are never changed.
3. Brings the four target groups in line with the table using exclusive roles, and changes only the differences.

| Role | Groups |
|---|---|
| 1 Community Member | Followers + Community Members |
| 2 Stakeholder | Followers + Stakeholders |
| 3 Contributor | Followers + Contributors |

Roles are managed **only** in the role table. Memberships added by hand for users who are in the table are undone on the next run. Group owners, group managers and anyone not in the table are never removed.

Set `DRY_RUN = True` to print the planned changes without writing. The scheduled task runs with `DRY_RUN = False`. See `README.md` next to this notebook.

## Configuration

In [ ]:
DRY_RUN = True

# Hub community organization. The password is set in ArcGIS Online only; never commit it.
HUB_PORTAL_URL = "https://hqfao-hub.maps.arcgis.com/"
HUB_ORG_ID = "D5aXW6TZFpeM2wke"
HUB_USERNAME = "hahmad_hqfao-hub"
HUB_PASSWORD = ""

# Source: every member of the Hub community organization joins this group automatically.
HUB_MEMBERS_GROUP_ID = "fb190ad8c9aa4d91aba380df83f341a6"

# Role table in the FAO organization: OBJECTID, fullName, email, username, role.
ROLE_TABLE_ITEM_ID = "7ecd39a2d8d040ba848b7e03036fe607"
COMMUNITY_MEMBER, STAKEHOLDER, CONTRIBUTOR = 1, 2, 3
VALID_ROLES = {COMMUNITY_MEMBER, STAKEHOLDER, CONTRIBUTOR}

# Target groups. "portal" names the connection that owns and edits the group.
# roles=None means every user in the table.
TARGET_GROUPS = {
    "followers": {
        "id": "3581cdd013a048e1b69a12fdf4cf186f",
        "portal": "fao",
        "expected_title": "FAO Data in Emergencies Hub Followers",
        "roles": None,
    },
    "community_members": {
        "id": "c8ae74a0f2de480abe6f72876a52b0cc",
        "portal": "fao",
        "expected_title": "Data In Emergency Hub - Community Members",
        "roles": {COMMUNITY_MEMBER},
    },
    "stakeholders": {
        "id": "568d613416b842bc8574259d0555829e",
        "portal": "fao",
        "expected_title": "Data In Emergency Hub - Stakeholders",
        "roles": {STAKEHOLDER},
    },
    "contributors": {
        "id": "ad13b87919464cb6b9bb6cd8defa0257",
        "portal": "fao",
        "expected_title": "Data In Emergency Hub - Contributors",
        "roles": {CONTRIBUTOR},
    },
}

BATCH_SIZE = 25                  # ArcGIS limit for add_users / remove_users
TABLE_EDIT_BATCH_SIZE = 500
BULK_DETAILS_THRESHOLD = 100     # above this many new users, read details with one paged search
FORCE_BULK_DETAILS = False       # True in a dry run to test the paged search path

# Destructive safety: refuse table deletions and group removals when the Hub group
# looks more than MAX_SHRINK_FRACTION *and* MAX_SHRINK_ABSOLUTE smaller than the table.
MAX_SHRINK_FRACTION = 0.10
MAX_SHRINK_ABSOLUTE = 100
ALLOW_LARGE_REMOVAL = False      # temporary override for a genuine mass offboarding

SAMPLE_SIZE = 10                 # usernames printed per change list

## Connect

In [ ]:
import time
from collections import defaultdict

from arcgis.gis import GIS

started = time.perf_counter()

gis = GIS("home")
gis_comm = GIS(HUB_PORTAL_URL, HUB_USERNAME, HUB_PASSWORD)
connections = {"fao": gis, "hub": gis_comm}
print(f"FAO: {gis.properties.user.username} | Hub: {gis_comm.properties.user.username} | DRY_RUN={DRY_RUN}")

## Planning functions

Pure functions with no ArcGIS calls, so the self-test cell can check them.

In [ ]:
def chunks(values, size):
    values = list(values)
    return [values[i:i + size] for i in range(0, len(values), size)]


def member_usernames(members):
    """Owner, admins and users from Group.get_members(), as one set."""
    names = set(members.get("admins") or []) | set(members.get("users") or [])
    if members.get("owner"):
        names.add(members["owner"])
    return names


def protected_usernames(members):
    """Group owner and managers are never removed."""
    names = set(members.get("admins") or [])
    if members.get("owner"):
        names.add(members["owner"])
    return names


def table_issues(rows, oid_field):
    issues = {"blank_username": [], "invalid_role": [], "duplicate_username": []}
    seen = defaultdict(list)
    for row in rows:
        username = row.get("username")
        if not username:
            issues["blank_username"].append(row.get(oid_field))
            continue
        seen[username].append(row)
        if row.get("role") not in VALID_ROLES:
            issues["invalid_role"].append(username)
    issues["duplicate_username"] = sorted(u for u, r in seen.items() if len(r) > 1)
    return issues


def plan_table_changes(hub_usernames, rows, oid_field):
    table_usernames = {r["username"] for r in rows if r.get("username")}
    new_usernames = hub_usernames - table_usernames
    departed_usernames = table_usernames - hub_usernames
    departed_oids = sorted(r[oid_field] for r in rows if r.get("username") in departed_usernames)
    return new_usernames, departed_usernames, departed_oids


def roles_after_update(rows, new_usernames, departed_usernames):
    """Roles per username once the table update is applied. Duplicate rows keep every role."""
    roles = defaultdict(set)
    for row in rows:
        username = row.get("username")
        if username and username not in departed_usernames and row.get("role") in VALID_ROLES:
            roles[username].add(row["role"])
        elif username and username not in departed_usernames:
            roles.setdefault(username, set())
    for username in new_usernames:
        roles[username].add(COMMUNITY_MEMBER)
    return roles


def desired_memberships(roles_by_user, hub_usernames, group_config):
    desired = {}
    for key, config in group_config.items():
        wanted = config["roles"]
        desired[key] = {
            u for u, roles in roles_by_user.items()
            if u in hub_usernames and (wanted is None or roles & wanted)
        }
    return desired


def plan_group_changes(desired, current, managed, protected):
    """Add what is missing; remove only managed, unprotected users who should not be there."""
    plan = {}
    for key in desired:
        plan[key] = {
            "add": desired[key] - current[key],
            "remove": (current[key] & managed) - desired[key] - protected[key],
        }
    return plan


def large_shrink(table_count, departed_count):
    if table_count == 0:
        return False
    return departed_count > MAX_SHRINK_ABSOLUTE and departed_count / table_count > MAX_SHRINK_FRACTION


def sample(values):
    values = sorted(values)
    more = f" (+{len(values) - SAMPLE_SIZE} more)" if len(values) > SAMPLE_SIZE else ""
    return ", ".join(values[:SAMPLE_SIZE]) + more

## Self-test

Runs the planning functions on small made-up data. It must pass before anything is read or written.

In [ ]:
def _self_test():
    groups = {
        "followers": {"roles": None},
        "community_members": {"roles": {1}},
        "stakeholders": {"roles": {2}},
        "contributors": {"roles": {3}},
    }
    oid = "OBJECTID"
    rows = [
        {oid: 1, "username": "anna", "role": 1},
        {oid: 2, "username": "bruno", "role": 2},   # changed from 1 to 2 in the table
        {oid: 3, "username": "carla", "role": 3},
        {oid: 4, "username": "gone", "role": 1},    # left the Hub
    ]
    hub = {"anna", "bruno", "carla", "newbie", "owner"}
    rows.append({oid: 5, "username": "owner", "role": 1})

    new, departed, departed_oids = plan_table_changes(hub, rows, oid)
    assert new == {"newbie"} and departed == {"gone"} and departed_oids == [4]

    roles = roles_after_update(rows, new, departed)
    assert roles["newbie"] == {1} and "gone" not in roles

    desired = desired_memberships(roles, hub, groups)
    assert desired["followers"] == {"anna", "bruno", "carla", "newbie", "owner"}
    assert desired["community_members"] == {"anna", "newbie", "owner"}

    current = {
        "followers": {"anna", "bruno", "carla", "gone", "owner"},
        "community_members": {"anna", "bruno", "gone", "fao_staff", "owner"},
        "stakeholders": set(),
        "contributors": {"carla", "owner"},
    }
    protected = {k: {"owner"} for k in groups}
    managed = {r["username"] for r in rows} | new
    plan = plan_group_changes(desired, current, managed, protected)

    assert plan["followers"] == {"add": {"newbie"}, "remove": {"gone"}}
    # Role change: added to Stakeholders, removed from Community Members.
    assert plan["stakeholders"]["add"] == {"bruno"}
    assert plan["community_members"]["remove"] == {"bruno", "gone"}
    # FAO staff not in the table, and the group owner, are never removed.
    assert "fao_staff" not in plan["community_members"]["remove"]
    assert "owner" not in plan["contributors"]["remove"]
    # Nothing to do when everything already matches.
    assert plan_group_changes(desired, desired, managed, protected) == {
        k: {"add": set(), "remove": set()} for k in groups
    }
    # Batching respects the 25-user limit.
    assert [len(b) for b in chunks(range(26), 25)] == [25, 1]
    # Shrink guard needs both thresholds.
    assert not large_shrink(3000, 50) and not large_shrink(2000, 101) and large_shrink(1000, 150)


_self_test()
print("Self-test passed.")

## Resolve groups and table by ID

In [ ]:
def resolve_group(conn, group_id, expected_title):
    group = conn.groups.get(group_id)
    if group is None:
        raise RuntimeError(f"Group {group_id} not found or not visible to {conn.properties.user.username}.")
    if expected_title and group.title.casefold() != expected_title.casefold():
        raise RuntimeError(f"Group {group_id} is titled '{group.title}', expected '{expected_title}'.")
    return group


hub_members_group = resolve_group(gis_comm, HUB_MEMBERS_GROUP_ID, None)
target_groups = {
    key: resolve_group(connections[c["portal"]], c["id"], c["expected_title"])
    for key, c in TARGET_GROUPS.items()
}
for key, group in target_groups.items():
    print(f"{key:18} {group.id}  '{group.title}'  owner={group.owner}")

role_table = gis.content.get(ROLE_TABLE_ITEM_ID).tables[0]
OID_FIELD = role_table.properties.objectIdField

## Read current state

In [ ]:
hub_members = hub_members_group.get_members()
hub_usernames = member_usernames(hub_members)

rows = [
    f.attributes for f in role_table.query(
        where="1=1",
        out_fields=f"{OID_FIELD},username,role",
        return_geometry=False,
    ).features
]

current_members, protected = {}, {}
for key, group in target_groups.items():
    members = group.get_members()
    current_members[key] = member_usernames(members)
    protected[key] = protected_usernames(members)

issues = table_issues(rows, OID_FIELD)
table_usernames = {r["username"] for r in rows if r.get("username")}
print(f"Hub members: {len(hub_usernames)} | table rows: {len(rows)} ({len(table_usernames)} usernames)")
for key in target_groups:
    print(f"  {key:18} {len(current_members[key])} members")
for name, values in issues.items():
    if values:
        print(f"Table issue - {name}: {len(values)}: {sample(str(v) for v in values)}")

## Plan and safety checks

In [ ]:
failures = []

if not hub_usernames:
    raise RuntimeError("The Hub community group returned no members; refusing to continue.")

new_usernames, departed_usernames, departed_oids = plan_table_changes(hub_usernames, rows, OID_FIELD)

destructive_allowed = True
if large_shrink(len(table_usernames), len(departed_usernames)) and not ALLOW_LARGE_REMOVAL:
    destructive_allowed = False
    failures.append("safety stop: large removal skipped (set ALLOW_LARGE_REMOVAL = True if genuine)")
    print(
        f"SAFETY STOP: {len(departed_usernames)} of {len(table_usernames)} table users are no longer "
        "in the Hub group. Table deletions and group removals are skipped. If this offboarding is "
        "genuine, rerun once with ALLOW_LARGE_REMOVAL = True."
    )

def departure_status(username):
    """'gone' (account deleted), 'left' (account exists, not in the Hub group),
    'member' (the group read missed them) or 'unknown'."""
    try:
        user = gis_comm.users.get(username)
    except Exception:
        return "unknown"
    if user is None:
        return "gone"
    groups = getattr(user, "groups", None)
    if groups is None:
        return "unknown"
    ids = {getattr(g, "id", None) or (g.get("id") if isinstance(g, dict) else None) for g in groups}
    return "member" if HUB_MEMBERS_GROUP_ID in ids else "left"


# Deleting a row loses the user's role, so each departure is confirmed on the user's own
# record. A user the group read missed means the read was incomplete: nothing destructive runs.
if departed_usernames and destructive_allowed:
    statuses = defaultdict(list)
    for username in sorted(departed_usernames):
        statuses[departure_status(username)].append(username)
    print("Departures: " + ", ".join(f"{k} {len(v)}" for k, v in sorted(statuses.items())))
    if statuses["member"] or statuses["unknown"]:
        destructive_allowed = False
        doubtful = statuses["member"] + statuses["unknown"]
        failures.append(
            f"safety stop: {len(doubtful)} departures not confirmed ({sample(doubtful)}); "
            "the Hub group read may be incomplete"
        )
        print("SAFETY STOP: departures not confirmed; table deletions and group removals are skipped.")

roles_by_user = roles_after_update(rows, new_usernames, departed_usernames)
desired = desired_memberships(roles_by_user, hub_usernames, TARGET_GROUPS)
managed = table_usernames | new_usernames   # before and after the table update
group_plan = plan_group_changes(desired, current_members, managed, protected)

print(f"Table: +{len(new_usernames)} new, -{len(departed_oids)} rows for {len(departed_usernames)} departed users")
if new_usernames:
    print(f"  new: {sample(new_usernames)}")
if departed_usernames:
    print(f"  departed: {sample(departed_usernames)}")
for key, change in group_plan.items():
    print(f"{key:18} +{len(change['add'])} -{len(change['remove'])}")
    if change["add"]:
        print(f"  add: {sample(change['add'])}")
    if change["remove"]:
        print(f"  remove: {sample(change['remove'])}")

## Update the role table

Details are looked up in dry runs too, so the lookup is tested before it is relied on.

In [ ]:
def user_details(usernames):
    """fullName and email for new users only. Per-user reads for a few, one paged search for many."""
    details = {}
    if FORCE_BULK_DETAILS or len(usernames) > BULK_DETAILS_THRESHOLD:
        for user in gis_comm.users.search(max_users=10000):
            if user.username in usernames:
                details[user.username] = user
    else:
        for username in usernames:
            user = gis_comm.users.get(username)
            if user is not None:
                details[username] = user
    return {
        u: {
            "fullName": getattr(details.get(u), "fullName", None),
            "email": getattr(details.get(u), "email", None),
        }
        for u in usernames
    }


def apply_table_edit(kind, **edits):
    """One edit_features call; returns the number of successful rows. Errors are recorded, not raised."""
    try:
        results = role_table.edit_features(**edits)[f"{kind}Results"]
    except Exception as error:
        failures.append(f"table {kind} batch error: {error}")
        return 0
    failed = [r for r in results if not r.get("success")]
    if failed:
        failures.append(f"table {kind}: {len(failed)} failed, first error {failed[0].get('error')}")
    return len(results) - len(failed)


details = {}
if new_usernames:
    try:
        details = user_details(new_usernames)
    except Exception as error:
        failures.append(f"user details lookup error: {error}")
    missing_name = sum(1 for d in details.values() if not d["fullName"])
    missing_email = sum(1 for d in details.values() if not d["email"])
    print(f"Details for {len(details)}/{len(new_usernames)} new users: {missing_name} without name, {missing_email} without email")

table_ok = not (new_usernames and not details)
if DRY_RUN:
    print("DRY_RUN: table not edited.")
else:
    if details:
        adds = [
            {"attributes": {"fullName": d["fullName"], "email": d["email"], "username": u, "role": COMMUNITY_MEMBER}}
            for u, d in sorted(details.items())
        ]
        added = sum(apply_table_edit("add", adds=batch) for batch in chunks(adds, TABLE_EDIT_BATCH_SIZE))
        print(f"Table rows added: {added}/{len(adds)}")
        table_ok &= added == len(adds)

    if departed_oids and destructive_allowed:
        deleted = sum(
            apply_table_edit("delete", deletes=",".join(str(oid) for oid in batch))
            for batch in chunks(departed_oids, TABLE_EDIT_BATCH_SIZE)
        )
        print(f"Table rows deleted: {deleted}/{len(departed_oids)}")
        table_ok &= deleted == len(departed_oids)

## Update group memberships

Additions run first. Removals run only when the table update succeeded and the safety checks allow it, and never touch a user whose addition failed in this run.

In [ ]:
def write_members(group, usernames, action):
    """add_users / remove_users in batches of 25; returns the usernames ArcGIS did not accept."""
    rejected = set()
    for batch in chunks(sorted(usernames), BATCH_SIZE):
        try:
            if action == "add":
                result = group.add_users(usernames=batch)
                rejected |= set(result.get("notAdded") or [])
            else:
                result = group.remove_users(usernames=batch)
                rejected |= set(result.get("notRemoved") or [])
        except Exception as error:
            failures.append(f"{group.title} {action} batch error: {error}")
            rejected |= set(batch)
    return rejected


if DRY_RUN:
    print("DRY_RUN: groups not edited.")
else:
    failed_adds = set()
    for key, change in group_plan.items():
        if change["add"]:
            rejected = write_members(target_groups[key], change["add"], "add")
            failed_adds |= rejected
            if rejected:
                failures.append(f"{key}: {len(rejected)} not added: {sample(rejected)}")
            print(f"{key:18} added {len(change['add']) - len(rejected)}/{len(change['add'])}")

    if not (table_ok and destructive_allowed):
        print("Removals skipped: table update failed or safety stop active.")
    else:
        for key, change in group_plan.items():
            to_remove = change["remove"] - failed_adds
            if to_remove:
                rejected = write_members(target_groups[key], to_remove, "remove")
                if rejected:
                    failures.append(f"{key}: {len(rejected)} not removed: {sample(rejected)}")
                print(f"{key:18} removed {len(to_remove) - len(rejected)}/{len(to_remove)}")

## Summary

In [ ]:
elapsed = time.perf_counter() - started
print(f"Finished in {elapsed:.0f}s | DRY_RUN={DRY_RUN} | failures={len(failures)}")
if failures:
    for failure in failures:
        print(" -", failure)
    raise RuntimeError(f"{len(failures)} problem(s) during the community sync; see the list above.")